# Ejercicio 3: Ruta Más Corta (Algoritmo de Dijkstra)
**Problema:** Determinar la ruta más corta entre los nodos **1** y **9** de la red dirigida.

In [1]:
import pandas as pd
from IPython.display import display, Markdown

# Definición del grafo dirigido (Nodo origen: {Nodo destino: Peso})
grafo = {
    1: {2: 4, 3: 2},
    2: {3: 2, 4: 7},
    3: {2: 4, 4: 9, 5: 6},
    4: {6: 1, 7: 5},
    5: {4: 2, 6: 4, 7: 3, 8: 2},
    6: {7: 1, 9: 5},
    7: {6: 4, 8: 3, 9: 5},
    8: {7: 2, 9: 6},
    9: {}
}

In [2]:
def dijkstra_iteraciones(grafo, inicio, fin):
    nodos = list(grafo.keys())
    dist = {n: float('inf') for n in nodos}
    pred = {n: '-' for n in nodos}
    dist[inicio] = 0
    fijos = set()
    
    def mostrar_tabla(iteracion, titulo, detalle=""):
        display(Markdown(f"### Iteración {iteracion}: {titulo}"))
        if detalle:
            display(Markdown(detalle))
        datos = {
            n: [
                f"[{'∞' if dist[n] == float('inf') else dist[n]}, {pred[n]}]{'*' if n in fijos else ''}",
                "Fijo" if n in fijos else "Temporal"
            ]
            for n in nodos
        }
        df = pd.DataFrame(datos, index=["Etiqueta", "Estado"])
        display(df)
    
    # Iteración 0: Inicialización
    mostrar_tabla(0, "Inicialización", "Se asigna distancia `0` al nodo origen (`1`) y `∞` al resto.")
    
    iteracion = 0
    while len(fijos) < len(nodos):
        u = min((n for n in nodos if n not in fijos), key=lambda n: dist[n])
        fijos.add(u)
        iteracion += 1
        
        evaluaciones = []
        for v, peso in grafo[u].items():
            if v not in fijos:
                nueva_dist = dist[u] + peso
                dist_actual = '∞' if dist[v] == float('inf') else dist[v]
                if nueva_dist < dist[v]:
                    dist[v] = nueva_dist
                    pred[v] = u
                    evaluaciones.append(f"- `{u} → {v}`: {dist[u]} + {peso} = **{nueva_dist}** < {dist_actual} ⇒ Se actualiza a `[{nueva_dist}, {u}]`")
                else:
                    evaluaciones.append(f"- `{u} → {v}`: {dist[u]} + {peso} = {nueva_dist} ≥ {dist_actual} ⇒ Se conserva `[{dist_actual}, {pred[v]}]`")
        
        detalle = f"**Nodo seleccionado:** `{u}` (distancia = `{dist[u]}`). Se marca como permanente.\n" + "\n".join(evaluaciones)
        mostrar_tabla(iteracion, f"Evaluación desde el Nodo {u}", detalle)
        
    return dist

distancias_minimas = dijkstra_iteraciones(grafo, 1, 9)

### Iteración 0: Inicialización

Se asigna distancia `0` al nodo origen (`1`) y `∞` al resto.

,1,2,3,4,5,6,7,8,9
Etiqueta,"[0, -]","[∞, -]","[∞, -]","[∞, -]","[∞, -]","[∞, -]","[∞, -]","[∞, -]","[∞, -]"
Estado,Temporal,Temporal,Temporal,Temporal,Temporal,Temporal,Temporal,Temporal,Temporal


### Iteración 1: Evaluación desde el Nodo 1

**Nodo seleccionado:** `1` (distancia = `0`). Se marca como permanente.
- `1 → 2`: 0 + 4 = **4** < ∞ ⇒ Se actualiza a `[4, 1]`
- `1 → 3`: 0 + 2 = **2** < ∞ ⇒ Se actualiza a `[2, 1]`

,1,2,3,4,5,6,7,8,9
Etiqueta,"[0, -]*","[4, 1]","[2, 1]","[∞, -]","[∞, -]","[∞, -]","[∞, -]","[∞, -]","[∞, -]"
Estado,Fijo,Temporal,Temporal,Temporal,Temporal,Temporal,Temporal,Temporal,Temporal


... (20 outputs left out)

,1,2,3,4,5,6,7,8,9
Etiqueta,"[0, -]*","[4, 1]*","[2, 1]*","[10, 5]*","[8, 3]*","[11, 4]*","[11, 5]*","[10, 5]*","[16, 8]"
Estado,Fijo,Fijo,Fijo,Fijo,Fijo,Fijo,Fijo,Fijo,Temporal


### Iteración 9: Evaluación desde el Nodo 9

**Nodo seleccionado:** `9` (distancia = `16`). Se marca como permanente.


,1,2,3,4,5,6,7,8,9
Etiqueta,"[0, -]*","[4, 1]*","[2, 1]*","[10, 5]*","[8, 3]*","[11, 4]*","[11, 5]*","[10, 5]*","[16, 8]*"
Estado,Fijo,Fijo,Fijo,Fijo,Fijo,Fijo,Fijo,Fijo,Fijo


In [3]:
def encontrar_rutas_optimas(grafo, inicio, fin, dist):
    rutas = []
    def backtrack(actual, camino):
        if actual == inicio:
            rutas.append(list(reversed(camino)))
            return
        for u, vecinos in grafo.items():
            if actual in vecinos and dist[u] + vecinos[actual] == dist[actual]:
                backtrack(u, camino + [u])
    backtrack(fin, [fin])
    return rutas

rutas = encontrar_rutas_optimas(grafo, 1, 9, distancias_minimas)
print(f"Distancia mínima total del nodo 1 al nodo 9: {distancias_minimas[9]}\n")
print("Rutas óptimas encontradas:")
for i, r in enumerate(rutas, 1):
    pesos = [grafo[r[j]][r[j+1]] for j in range(len(r)-1)]
    print(f"  Ruta {i}: {' -> '.join(map(str, r))} | Costo = {' + '.join(map(str, pesos))} = {sum(pesos)}")

Distancia mínima total del nodo 1 al nodo 9: 16

Rutas óptimas encontradas:
  Ruta 1: 1 -> 3 -> 5 -> 4 -> 6 -> 9 | Costo = 2 + 6 + 2 + 1 + 5 = 16
  Ruta 2: 1 -> 3 -> 5 -> 7 -> 9 | Costo = 2 + 6 + 3 + 5 = 16
  Ruta 3: 1 -> 3 -> 5 -> 8 -> 9 | Costo = 2 + 6 + 2 + 6 = 16
